# archive

> Read-only access to the reconcile-archive checkout

In [ ]:
#| default_exp archive

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re, html
from pathlib import Path
import markdown as md

In [ ]:
# test fixture: a throwaway archive matching the status.md contract in the spec
import tempfile
from pathlib import Path

STATUS_MD = """# {m} — receipt status

| date | title | amount | status | receipt_file | note |
|---|---|---|---|---|---|
| {m}-29 | OPENAI *CHATGPT SUBSCR | -110.46 | collected | [{m}-29_openai.pdf](receipts/{m}-29_openai.pdf) | src=x.pdf |
| {m}-06 | NORDEA | -20.46 | not-needed |  |  |

**missing (retrievable from Gmail/portal/Nordea): 1**
"""

def make_archive(months=('2025-07', '2025-08')):
    root = Path(tempfile.mkdtemp())
    (root/'README.md').write_text('readme')
    (root/'notamonth').mkdir()
    for m in months:
        d = root/m
        (d/'receipts').mkdir(parents=True)
        (d/'statement.pdf').write_bytes(b'%PDF-1.4 fake')
        (d/'statement.csv').write_text('date;amount\n')
        (d/'status.md').write_text(STATUS_MD.format(m=m))
        (d/'receipts'/f'{m}-29_openai.pdf').write_bytes(b'%PDF-1.4 fake receipt')
    return root

In [ ]:
#| export
MONTH_RE = re.compile(r'^\d{4}-\d{2}$')

def list_months(
    archive_dir: str|Path,  # path to the reconcile-archive checkout
) -> list[str]:             # sorted month names like '2025-07'
    "Month directories (YYYY-MM containing a status.md) in sorted order"
    return sorted(p.name for p in Path(archive_dir).iterdir()
                  if p.is_dir() and MONTH_RE.match(p.name) and (p/'status.md').exists())

In [ ]:
root = make_archive()
assert list_months(root) == ['2025-07', '2025-08']
(root/'2030-01').mkdir()  # month-shaped dir without status.md -> excluded
assert list_months(root) == ['2025-07', '2025-08']

In [ ]:
#| export
MISSING_RE = re.compile(r'\*\*missing[^:]*:\s*(\d+)\*\*')

def month_counts(
    archive_dir: str|Path,  # archive root
    month: str,             # month name like '2025-07'
) -> dict:                  # raw status tokens -> count, plus 'missing' from the footer line
    "Tally the status column of status.md; read missing from the footer"
    text = (Path(archive_dir)/month/'status.md').read_text()
    counts = {}
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith('|'): continue
        cells = [c.strip() for c in line.strip('|').split('|')]
        if len(cells) < 4: continue
        tok = cells[3]
        if not tok or tok.lower() == 'status' or set(tok) <= {'-'}: continue  # header/separator rows
        counts[tok] = counts.get(tok, 0) + 1
    m = MISSING_RE.search(text)
    if m is None: raise ValueError(f"{month}/status.md: missing-count footer line not found")
    counts['missing'] = int(m.group(1))
    return counts

In [ ]:
root = make_archive()
assert month_counts(root, '2025-07') == {'collected': 1, 'not-needed': 1, 'missing': 1}
# unknown tokens surface under their own key, never silently dropped
sm = root/'2025-08'/'status.md'
sm.write_text(sm.read_text().replace('not-needed', 'wtf-token'))
c = month_counts(root, '2025-08')
assert c['wtf-token'] == 1 and 'not-needed' not in c
# malformed status.md (no footer line) fails closed, never a silent missing=0
sm.write_text('\n'.join(l for l in sm.read_text().splitlines() if not l.startswith('**missing')))
try: month_counts(root, '2025-08'); assert False, 'should have raised'
except ValueError: pass

In [ ]:
#| export
RECEIPT_LINK_RE = re.compile(r'\]\(receipts/([A-Za-z0-9._-]+\.pdf)\)')
HREF_RE = re.compile(r'href="(?!/m/)[^"]*"')  # any rendered href not pointing at an authed route

def status_html(
    archive_dir: str|Path,  # archive root
    month: str,             # month name like '2025-07'
) -> str:                   # HTML fragment of the rendered status.md
    "Render status.md to HTML: rewrite receipt links, escape raw HTML, allowlist hrefs"
    text = (Path(archive_dir)/month/'status.md').read_text()
    text = RECEIPT_LINK_RE.sub(rf'](/m/{month}/receipt/\1)', text)
    out = md.markdown(html.escape(text, quote=False), extensions=['tables'])
    return HREF_RE.sub('', out)  # Markdown [x](javascript:…) survives escaping; only /m/… hrefs pass

In [ ]:
root = make_archive()
h = status_html(root, '2025-07')
assert '/m/2025-07/receipt/2025-07-29_openai.pdf' in h
assert '](receipts/' not in h and 'href="receipts/' not in h
assert '<table>' in h                       # tables extension active
assert 'not-needed' in h                    # rest of the Markdown intact
# raw HTML must be escaped, not passed through
sm = root/'2025-07'/'status.md'
sm.write_text(sm.read_text() + '\n<script>alert(1)</script>\n')
h = status_html(root, '2025-07')
assert '<script>' not in h and '&lt;script&gt;' in h
# Markdown link syntax survives HTML escaping — the href allowlist must strip it
sm.write_text(sm.read_text() + '\n[evil](javascript:alert(1))\n')
h = status_html(root, '2025-07')
assert 'javascript:' not in h
assert 'href="/m/2025-07/receipt/2025-07-29_openai.pdf"' in h  # receipt hrefs survive the allowlist

In [ ]:
#| export
def safe_file(
    archive_dir: str|Path,  # archive root
    month: str,             # month name like '2025-07'
    kind: str,              # 'statement_pdf' | 'statement_csv' | 'receipt'
    name: str|None = None,  # receipt filename (receipt kind only)
) -> Path:                  # resolved file inside the month dir
    "Resolve a servable file; raise FileNotFoundError on traversal or anything missing"
    if not MONTH_RE.match(month): raise FileNotFoundError(month)
    mdir = (Path(archive_dir)/month).resolve()
    if kind == 'statement_pdf':   p = mdir/'statement.pdf'
    elif kind == 'statement_csv': p = mdir/'statement.csv'
    elif kind == 'receipt':
        if not name or name != Path(name).name or not name.endswith('.pdf'):
            raise FileNotFoundError(name)
        p = mdir/'receipts'/name
    else: raise FileNotFoundError(kind)
    try:
        rp = p.resolve()  # follows symlinks; an escaping symlink lands outside mdir
        ok = rp.is_file() and rp.is_relative_to(mdir)
    except (ValueError, RuntimeError, OSError):  # null bytes, symlink loops, ...
        raise FileNotFoundError(p) from None
    if not ok: raise FileNotFoundError(p)
    return rp

In [ ]:
root = make_archive()

def rejects(*a):
    try: safe_file(*a)
    except FileNotFoundError: return True
    return False

assert safe_file(root, '2025-07', 'statement_pdf').name == 'statement.pdf'
assert safe_file(root, '2025-07', 'statement_csv').is_file()
assert safe_file(root, '2025-07', 'receipt', '2025-07-29_openai.pdf').is_file()
assert rejects(root, '2025-07', 'receipt', '../statement.pdf')      # traversal
assert rejects(root, '2025-07', 'receipt', '/etc/hostname')         # absolute
assert rejects(root, '2025-07', 'receipt', 'nope.pdf')              # missing
(root/'2025-07'/'receipts'/'secret.txt').write_text('x')
assert rejects(root, '2025-07', 'receipt', 'secret.txt')            # exists but not a .pdf
assert rejects(root, '2025-07', 'receipt', None)                    # no name
assert rejects(root, '..', 'statement_pdf')                         # bad month
assert rejects(root, '2025-07', 'bogus_kind')                       # bad kind
(root/'2025-07'/'receipts'/'evil.pdf').symlink_to('/etc/hostname')  # symlink escape
assert rejects(root, '2025-07', 'receipt', 'evil.pdf')
assert rejects(root, '2025-07', 'receipt', 'evil\x00.pdf')          # embedded null byte
(root/'2025-07'/'receipts'/'loop-a.pdf').symlink_to(root/'2025-07'/'receipts'/'loop-b.pdf')
(root/'2025-07'/'receipts'/'loop-b.pdf').symlink_to(root/'2025-07'/'receipts'/'loop-a.pdf')
assert rejects(root, '2025-07', 'receipt', 'loop-a.pdf')            # symlink loop

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()